In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torch.optim as optim
from scipy.stats import norm
import os

from model import *
from data import generate_gmm_data
from train_utils import train

from plot import *
from utils import *

import copy


In [ ]:
device = 'cuda:0' if torch.cuda.is_available() else 'cpu'
print(device)

In [ ]:
relu = lambda X: np.maximum(X, 0)
d_relu = lambda X: np.maximum(np.sign(X), 0)

elu = lambda X, alpha=1.0: ( alpha * (np.exp(X) - 1) ) * (X <= 0) + (X * (X > 0))
d_elu = lambda X, alpha=1.0: ( alpha * np.exp(X) ) * (X <= 0) + (X > 0)

gelu = lambda X: X * norm.cdf(X)
d_gelu = lambda X: norm.cdf(X) + X * norm.pdf(X)

sigmoid = lambda X: 1 / (1 + np.exp(-X))
d_sigmoid = lambda X: sigmoid(X) * (1 - sigmoid(X))

silu = lambda X: X * sigmoid(X)
d_silu = lambda X: sigmoid(X) + X * d_sigmoid(X)

leaky_relu = lambda X, slope=0.01: slope * np.minimum(X, 0) + np.maximum(X, 0)
d_leaky_relu = lambda X, slope=0.01: slope * (X <= 0) + (X > 0).astype(float)

identity = lambda X: X
d_identity = lambda X: np.ones(X.shape)

abs_ = lambda X: np.abs(X)
d_abs_ = lambda X: -1 * (X <= 0) + (X > 0)

softplus = lambda X: np.log(1 + np.exp(X))
d_softplus = lambda X: sigmoid(X)
d2_softplus = lambda X: d_sigmoid(X)

leaky_softplus = lambda X, slope=0.01: slope * X + (1 - slope) * (softplus(X) - np.log(2))
d_leaky_softplus = lambda X, slope=0.01: slope + (1 - slope) * (d_softplus(X))
d2_leaky_softplus = lambda X, slope = 0.01: (1 - slope) * d2_softplus(X)


smooth_abs = lambda X, beta = 10: (2 / beta) * softplus(X) - X - ( (2 / beta) * np.log(2) )
d_smooth_abs = lambda X, beta = 10: (2 / beta) * d_softplus(X) - 1

custom_activations_map = {'ReLU': [relu, d_relu],
               'Leaky-Softplus': [leaky_softplus, d_leaky_softplus], 
               'ELU': [elu, d_elu],
               'GELU': [gelu, d_gelu],
               'SiLU': [silu, d_silu],
               'Leaky-ReLU': [leaky_relu, d_leaky_relu],
               'Abs': [abs_, d_abs_],
               'SmoothAbs': [smooth_abs, d_smooth_abs],
               'Identity': [identity, d_identity]}

In [ ]:
optimizer_map = {'SGD': optim.SGD,
                'Adam': optim.Adam}

# Model Training

In [ ]:
def get_data(d: int, 
             K: int, 
             N_k: int, 
             scale: float = 1.0,
             nu: float = 1.0,
             normalize_: bool = False, 
             orthogonalize_: bool = False,
             whiten_: bool = False):
    
    
    N = K * N_k
    
    train_set, train_loader, _ = generate_gmm_data(d, 
                                                    K, 
                                                    N_k,
                                                    nu=nu,
                                                    data_scale = scale,
                                                    whiten_ = whiten_, 
                                                    normalize_ = normalize_)
    
    X = train_set.samples.T
    Y = train_set.targets.T
    
    return train_loader, X, Y

In [ ]:
def train_models(num_layers: int,
                data_dim: int, 
                hidden_dim: int, 
                num_classes: int, 
                act_fn_str: str, 
                train_loader, 
                val_loader, 
                epochs, 
                lr, 
                freq,
                loss_fn, 
                optimizer_type='SGD', 
                residual=False, 
                batchnorm=False, 
                init_method='orthogonal',
                last_layer_init_method='orthogonal',
                init_scale=1):
    
    
    # Initialize two models with identical initial weights, but 
    # keep classifier in one model frozen, and the other fixed. 
    model_frozen = MLP(num_layers, 
                data_dim, 
                hidden_dim, 
                num_classes, 
                act_fn_str, 
                freeze_last_layer=True,
                diff_init_last_layer=True,
                residual=residual, 
                batchnorm=batchnorm, 
                init_method=init_method, 
                last_layer_init_method=last_layer_init_method,
                init_scale=init_scale)
    
    model_unfrozen = copy.deepcopy(model_frozen) # Make deepcopy of model with frozen W2
    model_unfrozen.freeze_last_layer = False # Unfreeze last layer
    for param in model_unfrozen.layers[-1].parameters():
        param.requires_grad = True
    
    model_frozen = model_frozen.to(device)
    model_unfrozen = model_unfrozen.to(device)
    
    # Train model with frozen W2
    criterion_frozen = loss_map[loss_fn]
    optimizer_frozen = optimizer_map[optimizer_type](model_frozen.parameters(), lr=lr, weight_decay=0)
    results_frozen = train(model_frozen, 
                    criterion_frozen, 
                    optimizer_frozen, 
                    train_loader, 
                    val_loader, 
                    epochs, 
                    device,
                    scheduler=None, 
                    return_weights=True,
                    return_grads=True,
                    return_outs=False)

    weights_frozen = results_frozen['all_weights']
    grads_frozen = results_frozen['all_grads']
    
    # Train model with unfrozen W2
    criterion_unfrozen = loss_map[loss_fn]
    optimizer_unfrozen = optimizer_map[optimizer_type](model_unfrozen.parameters(), lr=lr, weight_decay=0)
    results_unfrozen = train(model_unfrozen, 
                    criterion_unfrozen, 
                    optimizer_unfrozen, 
                    train_loader, 
                    val_loader, 
                    epochs, 
                    device,
                    scheduler=None, 
                    return_weights=True,
                    return_grads=True,
                    return_outs=False)

    weights_unfrozen = results_unfrozen['all_weights']
    grads_unfrozen = results_unfrozen['all_grads']
    
    return weights_frozen, grads_frozen, weights_unfrozen, grads_unfrozen

In [ ]:
def fro_inner_product(X: np.ndarray, Y: np.ndarray):
    assert X.shape == Y.shape
    
    X_norm = np.linalg.norm(X, ord='fro')
    Y_norm = np.linalg.norm(Y, ord='fro')
    
    return np.trace(X.T @ Y) / (X_norm * Y_norm)

def normalized_dist(X: np.ndarray, Y: np.ndarray, norm_factor: float = 1.0):
    assert X.shape == Y.shape
    
    return np.linalg.norm(X - Y)**2 / norm_factor 

def normalized_norms(X: np.ndarray):
    return normalized_dist(X, np.zeros(X.shape))

def sin_theta_norms(U1: np.ndarray, U2: np.ndarray):
    svals = np.linalg.svd(U1.T @ U2, compute_uv=False)
    angles = np.arccos(np.clip(svals, 0, 1))
    
    return np.linalg.norm(np.sin(angles))

In [ ]:
reduction = 'sum'
loss_map = {
    'ce_loss': nn.CrossEntropyLoss(reduction=reduction),
    'mse_loss': nn.MSELoss(reduction=reduction)
}

data_type_map = {
    'gmm': 'GMM', 
    'uos': 'UoS',
    'random': 'Random'
}

In [ ]:
# Data setup
d = 64 # 128 # Ambient dim
m = 72 #192
K = 4 # Number of classes
N_k = 250
N = N_k * K
nu = np.sqrt(3)
scale = 1

# Model setup
L = 2
act_fn_strs = ['ELU', 'GELU', 'SiLU']
residual = False
batchnorm = False
freeze_last_layer = False
init_method = 'orthogonal'
last_layer_init_methods = ['gaussian']
init_scale = 1e-2

whiten_ = True
normalize_ = False
orthogonalize_ = False

# Training setup
epochs = 100
lr = 1e-2
freq = 1
optimizer_type = 'SGD'
#loss_fn = 'ce_loss'
loss_fn = 'mse_loss'

save = True


# Compare G1 subspaces when W2 is frozen vs. not frozen

In [ ]:
trials = 10

L_dists = np.zeros((trials, len(act_fn_strs), len(last_layer_init_methods), epochs))
R_dists = np.zeros((trials, len(act_fn_strs), len(last_layer_init_methods), epochs))
all_W2_svals = np.zeros((trials, len(act_fn_strs), len(last_layer_init_methods), m))
for i in range(trials):
    for j, act_fn_str in enumerate(act_fn_strs):
        for k, last_layer_init_method in enumerate(last_layer_init_methods):
        
            # Create data
            train_loader, X, Y = get_data(d, K, N_k, scale, nu, normalize_, orthogonalize_, whiten_)
            val_loader = train_loader

            print("=== TRIAL " + str(i) + " ===")

            # Train model
            weights_frozen, grads_frozen, weights_unfrozen, grads_unfrozen = \
                train_models(L, d, m, K, act_fn_str, 
                             train_loader, 
                             val_loader, 
                             epochs, 
                             lr, 
                             freq, 
                             loss_fn, 
                             optimizer_type=optimizer_type,
                             residual=residual,
                             batchnorm=batchnorm,
                             init_method=init_method,
                             last_layer_init_method=last_layer_init_method,
                             init_scale=init_scale)
            
            
            all_W2 = np.zeros((K*epochs, m))
            for t in range(epochs):
                G1_t_frozen = grads_frozen[t][0]
                G1_t_unfrozen = grads_unfrozen[t][0]
                
                L_f, _, R_f = np.linalg.svd(G1_t_frozen)
                L_uf, _, R_uf = np.linalg.svd(G1_t_unfrozen)
                
                R_f, R_uf = R_f.T, R_uf.T
                
                L1_f, R1_f = L_f[:, :K], R_f[:, :K]
                L1_uf, R1_uf = L_uf[:, :K], R_uf[:, :K]
                
                _, _, W2_t_rspace_h = np.linalg.svd(weights_unfrozen[t][1], full_matrices=False)
                W2_t_rspace = W2_t_rspace_h.T
                
                L_dists[i, j, k, t] = sin_theta_norms(L1_f, L1_uf)**2 / K
                R_dists[i, j, k, t] = sin_theta_norms(R1_f, R1_uf)**2 / K
                
                all_W2[t*K:(t+1)*K, :] = weights_unfrozen[t][1]
            
            
            all_W2_svals[i, j, k, :] = np.linalg.svd(all_W2, compute_uv=False)
                


In [ ]:
L_dists_mean = np.mean(L_dists, axis=0)
R_dists_mean = np.mean(R_dists, axis=0)

L_dists_std = np.std(L_dists, axis=0) / np.sqrt(trials)
R_dists_std = np.std(R_dists, axis=0) / np.sqrt(trials)

n_plots = len(act_fn_strs)
fig, axes = plt.subplots(1, n_plots, figsize=(6 * n_plots, 5), sharey=True)

if n_plots == 1:
    axes = [axes]

for j, (ax, act_fn_str) in enumerate(zip(axes, act_fn_strs)):
    for k, last_layer_init_method in enumerate(last_layer_init_methods):
        if j == 0:
            label_1, label_2 = 'Top-$K$ Left', 'Top-$K$ Right'
        else:
            label_1, label_2 = None, None
            
        l_line, = ax.plot(np.arange(epochs), L_dists_mean[j, k, :], label=label_1)
        ax.fill_between(np.arange(epochs),
                        L_dists_mean[j, k, :] + L_dists_std[j, k, :],
                        L_dists_mean[j, k, :] - L_dists_std[j, k, :],
                        alpha=0.5, color=l_line.get_color())

        r_line, = ax.plot(np.arange(epochs), R_dists_mean[j, k, :], label=label_2)
        ax.fill_between(np.arange(epochs),
                        R_dists_mean[j, k, :] + R_dists_std[j, k, :],
                        R_dists_mean[j, k, :] - R_dists_std[j, k, :],
                        alpha=0.5, color=r_line.get_color())

    ax.set_title(act_fn_str, fontsize=24) 
    ax.tick_params(axis='both', labelsize=24)
    ax.grid(True)


fig.suptitle(rf'Distance Between $G_1(t)$ Singular Subspaces for Frozen vs. Trained $W_2$', fontsize=30, x=0.53, y=1.1)
fig.supxlabel('Train Epoch', fontsize=30)
fig.supylabel('Normalized Squared\nSubspace Distance', fontsize=24, x=0)
fig.legend(fontsize=28, bbox_to_anchor=(0.75, 1.04), ncol=2, frameon=False)
plt.tight_layout()
if save:
    plt.savefig('./figs/rebuttal/grad_subspaces_W2_frozen_vs_trained.pdf', bbox_inches='tight')
plt.show()

# for j, act_fn_str in enumerate(act_fn_strs):
#     plt.figure()
    
#     for k, last_layer_init_method in enumerate(last_layer_init_methods):
#         plt.plot(np.arange(epochs), L_dists_mean[j, k, :], label='Left Subspace')
#         plt.fill_between(np.arange(epochs), L_dists_mean[j, k, :] + L_dists_std[j, k, :], L_dists_mean[j, k, :] - L_dists_std[j, k, :], alpha=0.5)
#         plt.plot(np.arange(epochs), R_dists_mean[j, k, :], label='Right Subspace')
#         plt.fill_between(np.arange(epochs), R_dists_mean[j, k, :] + R_dists_std[j, k, :], R_dists_mean[j, k, :] - R_dists_std[j, k, :], alpha=0.5)
    
#     plt.title(rf'Distance Between $G_1(t)$ Subspaces' + '\n' + 'Frozen vs. Trained $W_2$ (' + act_fn_str + ')', fontsize=18)
#     plt.xlabel('Train Epoch', fontsize=18)
#     plt.ylabel('Normalized Squared\nSubspace Distance', fontsize = 18)
#     plt.xticks(fontsize=18)
#     plt.yticks(fontsize=18)
#     plt.grid('on')
#     plt.legend(fontsize=16)
#     plt.show()
#     plt.close()

In [ ]:
all_W2_svals_mean = np.mean(all_W2_svals, axis=0)
all_W2_svals_std = np.std(all_W2_svals, axis=0)

plt.figure()
for j, act_fn_str in enumerate(act_fn_strs):
    for k, last_layer_init_method in enumerate(last_layer_init_methods):
        plt.errorbar(np.arange(all_W2_svals_mean.shape[-1]) + 1, all_W2_svals_mean[j, k, :], 
                     yerr=all_W2_svals_std[j, k, :], 
                     fmt='o', label=act_fn_str, markersize=8, capsize=10)

plt.title(r'Singular Values of Concatenated $W_2(t)$ ($K = ' + str(K) + r'$)', fontsize=18)
plt.xlabel('Singular Value Index', fontsize=18)
plt.ylabel('Singular Value', fontsize=18)
plt.xticks(fontsize=16)
plt.yticks(fontsize=16)
plt.legend(fontsize=16)
plt.grid('on')
if save:
    plt.savefig('./figs/rebuttal/concat_W2_svals.pdf', bbox_inches='tight')
plt.show()
plt.close()